# Retrieval v2 – Stage 7 Candidate Reranking

Evaluate the five frozen local multilingual rerankers on the exact Stage 6 candidate pool. The primary decision combines retrieval quality with reranking time, GPU memory, paired failures, and the predefined operational p90 latency ceiling.


In [11]:
from google.colab import drive
drive.mount("/content/drive")

import json, os, subprocess, sys
from pathlib import Path
import pandas as pd

CODE_ROOT     = Path("/content/drive/.shortcut-targets-by-id/1VXCKOD57uox_lUy-66Vqz5cKfzk8cvs4/aic26/asr_model_comparison")
ARTIFACT_ROOT = CODE_ROOT
CONFIG_PATH   = CODE_ROOT / "configs" / "retrieval_v2.json"
RUNNER_PATH   = CODE_ROOT / "src" / "07_evaluate_retrieval_v2.py"

os.environ["AIC_RETRIEVAL_ARTIFACT_ROOT"] = str(ARTIFACT_ROOT)
os.environ["AIC_RETRIEVAL_CONFIG"] = str(CONFIG_PATH)
os.chdir(CODE_ROOT)

print("Code root:    ", CODE_ROOT)
print("Artifact root:", ARTIFACT_ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Code root:     /content/drive/.shortcut-targets-by-id/1VXCKOD57uox_lUy-66Vqz5cKfzk8cvs4/aic26/asr_model_comparison
Artifact root: /content/drive/.shortcut-targets-by-id/1VXCKOD57uox_lUy-66Vqz5cKfzk8cvs4/aic26/asr_model_comparison


In [12]:
# Stage 7 uses the SentenceTransformers 5.4 CrossEncoder integrations used by Qwen3 and Mixedbread.
subprocess.run([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "--upgrade",
    "sentence-transformers==5.4.0",
    "transformers==4.52.0",
    "einops",
], check = True)
print("Reranker dependencies ready")

Reranker dependencies ready


In [13]:
import importlib.metadata

transformers_version = importlib.metadata.version("transformers")
sentence_transformers_version = importlib.metadata.version("sentence-transformers")

print("transformers:", transformers_version)
print("sentence-transformers:", sentence_transformers_version)

config = json.loads(CONFIG_PATH.read_text(encoding = "utf-8"))

decision = config["selection"]["stage06_decision"]
runtime  = config["stage07_reranking"]["runtime"]

latency_ceiling = runtime["operational_latency_ceiling_ms_p90"]

assert transformers_version == runtime["required_transformers_version"]
assert sentence_transformers_version == runtime["required_sentence_transformers_version"]

assert decision["status"] == "frozen_after_review", (
    "Freeze selection.stage06_decision after Stage 6 review first."
)

assert latency_ceiling is not None, (
    "Set stage07_reranking.runtime.operational_latency_ceiling_ms_p90 before Stage 7."
)

print("Frozen candidate policy:", decision["selected_policy_id"])
print("Candidate videos K:     ", decision["selected_video_k"])
print("Windows/video M:        ", decision["selected_windows_per_video"])
print("p90 latency ceiling ms: ", latency_ceiling)

print("Rerankers:", *config["stage07_reranking"]["models"].keys(), sep = "\n  ")

transformers: 4.52.0
sentence-transformers: 5.4.0
Frozen candidate policy: K30_M5
Candidate videos K:      30
Windows/video M:         5
p90 latency ceiling ms:  5000.0
Rerankers:
  R1_mmarco_minilm
  R2_gte_multilingual
  R3_bge_v2_m3
  R4_qwen3_0_6b
  R5_mxbai_base_v2
  R6_jina_v2_multilingual


In [14]:
command = [
    sys.executable,
    "-u",
    str(RUNNER_PATH),
    "--stage",
    "stage07_reranking",
]


environment = os.environ.copy()
environment["PYTHONUNBUFFERED"] = "1"


print("Running:", " ".join(command), flush = True)


process = subprocess.Popen(
    command,
    cwd = CODE_ROOT,
    env = environment,
    stdout = subprocess.PIPE,
    stderr = subprocess.STDOUT,
    text = True,
    bufsize = 1,
)


for line in process.stdout :
    print(line, end = "", flush = True)


return_code = process.wait()


if (return_code != 0) :
    raise subprocess.CalledProcessError(return_code, command)

Running: /usr/bin/python3 -u /content/drive/.shortcut-targets-by-id/1VXCKOD57uox_lUy-66Vqz5cKfzk8cvs4/aic26/asr_model_comparison/src/07_evaluate_retrieval_v2.py --stage stage07_reranking
AIC 2026 RETRIEVAL V2 – STAGE 7 CANDIDATE RERANKING
      R6_jina_v2_multilingual: running one-pair compatibility smoke test
2026-08-11 10:07:09.265061: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
      R6_jina_v2_multilingual: loading jinaai/jina-reranker-v2-base-multilingual@9cfeff2df7d40d1b78e75e5e9cebec92a99813c9 on cuda
      R6_jina_v2_multilingual: model ready in 5.0s; resolved revision 9cfeff2df7d40d1b78e75e5e9cebec92a99813c9
      R6_jina_v2_multilingual: smoke test PASS; score=0.808105
/content/drive/.shortcut-targets-by-id/1VXCKOD57uox_lUy-66Vqz5cK

In [15]:
REPORT_ROOT = ARTIFACT_ROOT / "reports" / "retrieval_v2" / "stage07_reranking"
summary = json.loads((REPORT_ROOT / "stage_summary.json").read_text(encoding = "utf-8"))
quality_latency = pd.read_csv(REPORT_ROOT / "quality_latency_summary.csv")

print("Stage 7 status:", "PASS" if summary["passed"] else "FAIL")
display(quality_latency.sort_values(["video_mrr", "p90_query_ms"], ascending = [False, True]))

Stage 7 status: PASS


,method_id,model_id,view,query_count,video_rr_mean,story_rr_mean,video_rr_delta_mean_vs_control,story_rr_delta_mean_vs_control,video_better_count,video_tie_count,...,mean_query_ms,p50_query_ms,p90_query_ms,p95_query_ms,pairs_per_second,amortized_ms_per_pair,total_input_tokens,tokens_per_second,operationally_eligible,pareto_optimal
7,R4_qwen3_0_6b__S2_first_stage_reranker_50_50,R4_qwen3_0_6b,processed,40,0.864194,0.862500,0.053149,-0.041667,8,28,...,6687.851428,6607.990175,7307.764522,7405.363581,22.320322,44.802220,2161379.0,8079.496918,False,True
5,R3_bge_v2_m3__S2_first_stage_reranker_50_50,R3_bge_v2_m3,processed,40,0.820524,0.912500,0.009479,0.008333,6,30,...,2127.255855,2027.758579,2538.644480,2589.968703,70.172565,14.250584,1889299.0,22203.476319,True,True
12,S0_first_stage,first_stage,processed,40,0.811045,0.904167,0.000000,0.000000,0,40,...,0.000000,0.000000,0.000000,0.000000,NaN,0.000000,0.0,0.000000,True,False
3,R2_gte_multilingual__S2_first_stage_reranker_5...,R2_gte_multilingual,processed,40,0.797247,0.898750,-0.013798,-0.005417,5,29,...,913.193712,881.781823,1042.447710,1084.714880,163.464770,6.117526,1889299.0,51722.295467,True,True
9,R5_mxbai_base_v2__S2_first_stage_reranker_50_50,R5_mxbai_base_v2,processed,40,0.795467,0.889583,-0.015578,-0.014583,4,29,...,4127.758848,4019.197745,4565.808907,4654.172340,36.163692,27.652044,2035839.0,12330.171618,True,False
4,R3_bge_v2_m3__S1_reranker_only,R3_bge_v2_m3,processed,40,0.783322,0.891667,-0.027723,-0.012500,5,27,...,2127.255855,2027.758579,2538.644480,2589.968703,70.172565,14.250584,1889299.0,22203.476319,True,False
11,R6_jina_v2_multilingual__S2_first_stage_rerank...,R6_jina_v2_multilingual,processed,40,0.754271,0.911250,-0.056774,0.007083,4,25,...,840.405098,806.398774,1016.586797,1049.846011,177.622673,5.629912,1889299.0,56202.032917,True,True
1,R1_mmarco_minilm__S2_first_stage_reranker_50_50,R1_mmarco_minilm,processed,40,0.715417,0.872917,-0.095628,-0.031250,5,25,...,278.127707,272.800902,335.985468,341.340114,536.713877,1.863190,1889299.0,169822.976138,True,True
6,R4_qwen3_0_6b__S1_reranker_only,R4_qwen3_0_6b,processed,40,0.701412,0.847917,-0.109633,-0.056250,6,21,...,6687.851428,6607.990175,7307.764522,7405.363581,22.320322,44.802220,2161379.0,8079.496918,False,False
8,R5_mxbai_base_v2__S1_reranker_only,R5_mxbai_base_v2,processed,40,0.692747,0.839583,-0.118298,-0.064583,4,22,...,4127.758848,4019.197745,4565.808907,4654.172340,36.163692,27.652044,2035839.0,12330.171618,True,False


In [16]:
preflight = pd.read_csv(REPORT_ROOT / "runtime_preflight.csv")
latency   = pd.read_csv(REPORT_ROOT / "latency_summary.csv")
tokens    = pd.read_csv(REPORT_ROOT / "token_length_summary.csv")

print("Runtime / compatibility preflight")
display(preflight)
print("Full40 reranking latency")
display(latency)
print("Token-length audit")
display(tokens)

Runtime / compatibility preflight


,backend_id,cache_hit,preflight_pairs,selected_batch_size,attempts,warmup_s,passed
0,R1_mmarco_minilm,False,741,32,"[{""batch_size"": 32, ""passed"": true, ""runtime_s...",0.028696,True
1,R2_gte_multilingual,False,741,32,"[{""batch_size"": 32, ""passed"": true, ""runtime_s...",0.047418,True
2,R3_bge_v2_m3,False,741,32,"[{""batch_size"": 32, ""passed"": true, ""runtime_s...",0.090087,True
3,R4_qwen3_0_6b,False,741,32,"[{""batch_size"": 32, ""passed"": true, ""runtime_s...",0.322322,True
4,R5_mxbai_base_v2,False,741,32,"[{""batch_size"": 32, ""passed"": true, ""runtime_s...",0.200916,True
5,R6_jina_v2_multilingual,False,741,32,"[{""batch_size"": 32, ""passed"": true, ""runtime_s...",0.045986,True


Full40 reranking latency


,backend_id,query_count,candidate_pairs_mean,candidate_pairs_max,total_pairs,total_runtime_s,mean_query_ms,pairs_per_second,amortized_ms_per_pair,p50_query_ms,p90_query_ms,p95_query_ms,total_input_tokens,tokens_per_second,operational_latency_ceiling_ms_p90,operationally_eligible
0,R1_mmarco_minilm,40,149.275,150,5971,11.125108,278.127707,536.713877,1.863190,272.800902,335.985468,341.340114,1889299,169822.976138,5000.0,True
1,R2_gte_multilingual,40,149.275,150,5971,36.527748,913.193712,163.464770,6.117526,881.781823,1042.447710,1084.714880,1889299,51722.295467,5000.0,True
2,R3_bge_v2_m3,40,149.275,150,5971,85.090234,2127.255855,70.172565,14.250584,2027.758579,2538.644480,2589.968703,1889299,22203.476319,5000.0,True
3,R4_qwen3_0_6b,40,149.275,150,5971,267.514057,6687.851428,22.320322,44.802220,6607.990175,7307.764522,7405.363581,2161379,8079.496918,5000.0,False
4,R5_mxbai_base_v2,40,149.275,150,5971,165.110354,4127.758848,36.163692,27.652044,4019.197745,4565.808907,4654.172340,2035839,12330.171618,5000.0,True
5,R6_jina_v2_multilingual,40,149.275,150,5971,33.616204,840.405098,177.622673,5.629912,806.398774,1016.586797,1049.846011,1889299,56202.032917,5000.0,True


Token-length audit


,backend_id,pair_count,total_input_tokens,mean_input_tokens,p50_input_tokens,p90_input_tokens,p95_input_tokens,max_input_tokens,max_length,truncated_count,truncated_fraction,audit_policy
0,R1_mmarco_minilm,5971,1889299,316.412494,315.0,375.0,396.0,467,512,0,0.000000,cross_encoder_preprocess_untruncated_v1
1,R2_gte_multilingual,5971,1889299,316.412494,315.0,375.0,396.0,467,512,0,0.000000,cross_encoder_preprocess_untruncated_v1
2,R3_bge_v2_m3,5971,1889299,316.412494,315.0,375.0,396.0,467,512,0,0.000000,cross_encoder_preprocess_untruncated_v1
3,R4_qwen3_0_6b,5971,2161379,361.979400,360.0,428.0,449.0,527,512,7,0.001172,cross_encoder_preprocess_untruncated_v1
4,R5_mxbai_base_v2,5971,2035839,340.954446,339.0,407.0,428.0,506,512,0,0.000000,cross_encoder_preprocess_untruncated_v1
5,R6_jina_v2_multilingual,5971,1889299,316.412494,315.0,375.0,396.0,467,512,0,0.000000,cross_encoder_preprocess_untruncated_v1


In [17]:
comparison = pd.read_csv(REPORT_ROOT / "paired_comparison.csv")
failures   = pd.read_csv(REPORT_ROOT / "failure_cases.csv")

print("Paired query behavior")
display(comparison.groupby("method_id")["video_relation"].value_counts().unstack(fill_value = 0))
print("Failure cases")
display(failures if len(failures) else pd.DataFrame({"status" : ["No Stage 7 failure cases"]}))

Paired query behavior


video_relation,better,tie,worse
method_id,,,
R1_mmarco_minilm__S1_reranker_only,5,10,25
R1_mmarco_minilm__S2_first_stage_reranker_50_50,5,25,10
R2_gte_multilingual__S1_reranker_only,5,22,13
R2_gte_multilingual__S2_first_stage_reranker_50_50,5,29,6
R3_bge_v2_m3__S1_reranker_only,5,27,8
R3_bge_v2_m3__S2_first_stage_reranker_50_50,6,30,4
R4_qwen3_0_6b__S1_reranker_only,6,21,13
R4_qwen3_0_6b__S2_first_stage_reranker_50_50,8,28,4
R5_mxbai_base_v2__S1_reranker_only,4,22,14


Failure cases


,query_set,corpus_id,method_id,model_id,view,query_id,query_text,query_category,task_type,difficulty,...,first_stage_story_rank,video_rank_delta,correct_video_retained,relevant_window_retained,joint_candidate_success,candidate_video_count,candidate_pair_count,large_regression,hard_top1_regression,failure_type
0,extension40_full,all50,R1_mmarco_minilm__S1_reranker_only,R1_mmarco_minilm,processed,R1-2,Mẩu tin giới thiệu về đàn hổ tại một địa phươn...,number_or_measurement,KIS,easy,...,1.0,1.0,True,True,True,30.0,150.0,False,False,RERANKER_FAILURE
1,extension40_full,all50,R1_mmarco_minilm__S1_reranker_only,R1_mmarco_minilm,processed,R1-9,Đoạn clip là cảnh thu hoạch dứa ở miền Tây: mộ...,person_organization_location,KIS,medium,...,4.0,14.0,True,True,True,30.0,150.0,True,True,RERANKER_FAILURE
2,extension40_full,all50,R1_mmarco_minilm__S1_reranker_only,R1_mmarco_minilm,processed,R1-13,Đoạn video mô tả một người ngồi vệ sinh máy ản...,other,KIS,medium,...,4.0,12.0,True,True,True,30.0,149.0,True,True,RERANKER_FAILURE
3,extension40_full,all50,R1_mmarco_minilm__S1_reranker_only,R1_mmarco_minilm,processed,R1-15,Đoạn video về một chương trình từ thiện của mộ...,person_organization_location,QA,easy,...,1.0,2.0,True,True,True,30.0,146.0,False,False,RERANKER_FAILURE
4,extension40_full,all50,R1_mmarco_minilm__S1_reranker_only,R1_mmarco_minilm,processed,R1-22,Đoạn video về một người phụ nữ dạy nấu ăn cho ...,number_or_measurement,QA,easy,...,1.0,23.0,True,True,True,30.0,150.0,True,True,RERANKER_FAILURE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
178,extension40_full,all50,R6_jina_v2_multilingual__S2_first_stage_rerank...,R6_jina_v2_multilingual,processed,R1-7,Đoạn clip cần tìm là cảnh quay trong một khu r...,other,KIS,medium,...,1.0,2.0,True,True,True,30.0,150.0,False,False,RERANKER_FAILURE
179,extension40_full,all50,R6_jina_v2_multilingual__S2_first_stage_rerank...,R6_jina_v2_multilingual,processed,R1-10,Tìm chính xác đoạn clip ngắn có ba người (hai ...,number_or_measurement,KIS,medium,...,1.0,4.0,True,True,True,30.0,150.0,False,False,RERANKER_FAILURE
180,extension40_full,all50,R6_jina_v2_multilingual__S2_first_stage_rerank...,R6_jina_v2_multilingual,processed,R2-18,Cảnh quay các con mèo trong một căn phòng. Có ...,other,KIS,easy,...,1.0,5.0,True,True,True,30.0,146.0,False,False,RERANKER_FAILURE
181,extension40_full,all50,R6_jina_v2_multilingual__S2_first_stage_rerank...,R6_jina_v2_multilingual,processed,R2-15,Một người đang chụp hình những chiếc ván trượt...,other,KIS,medium,...,1.0,1.0,True,True,True,30.0,149.0,False,False,RERANKER_FAILURE


In [18]:
pareto = quality_latency[quality_latency["pareto_optimal"] == True]  # noqa: E712
print("Pareto-optimal quality/latency candidates")
display(pareto)
print("No reranker is selected automatically. Review quality, p90 reranking time, memory, rank-1 recoveries/losses, and failure cases before freezing selection.stage07_decision.")

Pareto-optimal quality/latency candidates


,method_id,model_id,view,query_count,video_rr_mean,story_rr_mean,video_rr_delta_mean_vs_control,story_rr_delta_mean_vs_control,video_better_count,video_tie_count,...,mean_query_ms,p50_query_ms,p90_query_ms,p95_query_ms,pairs_per_second,amortized_ms_per_pair,total_input_tokens,tokens_per_second,operationally_eligible,pareto_optimal
1,R1_mmarco_minilm__S2_first_stage_reranker_50_50,R1_mmarco_minilm,processed,40,0.715417,0.872917,-0.095628,-0.031250,5,25,...,278.127707,272.800902,335.985468,341.340114,536.713877,1.863190,1889299.0,169822.976138,True,True
3,R2_gte_multilingual__S2_first_stage_reranker_5...,R2_gte_multilingual,processed,40,0.797247,0.898750,-0.013798,-0.005417,5,29,...,913.193712,881.781823,1042.447710,1084.714880,163.464770,6.117526,1889299.0,51722.295467,True,True
5,R3_bge_v2_m3__S2_first_stage_reranker_50_50,R3_bge_v2_m3,processed,40,0.820524,0.912500,0.009479,0.008333,6,30,...,2127.255855,2027.758579,2538.644480,2589.968703,70.172565,14.250584,1889299.0,22203.476319,True,True
7,R4_qwen3_0_6b__S2_first_stage_reranker_50_50,R4_qwen3_0_6b,processed,40,0.864194,0.862500,0.053149,-0.041667,8,28,...,6687.851428,6607.990175,7307.764522,7405.363581,22.320322,44.802220,2161379.0,8079.496918,False,True
11,R6_jina_v2_multilingual__S2_first_stage_rerank...,R6_jina_v2_multilingual,processed,40,0.754271,0.911250,-0.056774,0.007083,4,25,...,840.405098,806.398774,1016.586797,1049.846011,177.622673,5.629912,1889299.0,56202.032917,True,True


No reranker is selected automatically. Review quality, p90 reranking time, memory, rank-1 recoveries/losses, and failure cases before freezing selection.stage07_decision.
